# EDA notebook

# **Machine Learning project : Intellicanteen**

Our team :
- **OUADAH Lina Selma (G5)**
- **BELAMRI Chakib (G1)**
- **HENNI Mohammed Yassine (G1)**
- **MEZIGHECHE Malak Yasmine (G10)**


##  Notebook Overview

This notebook is the **first and mandatory step** of the IntelliCanteen ML pipeline.
It must be run in full **before** any of the modeling notebooks (`breakfast_pipeline`, `dinner_pipeline`, etc.).

### What this notebook produces
| Output file | Description |
|---|---|
| `dataset.xlsx` | Fully merged + cleaned + feature-engineered master dataset |
| `breakfast_data.xlsx` | Breakfast subset (meal_type_id = 1) |
| `lunch_data.xlsx` | Lunch subset (meal_type_id = 2) |
| `dinner_data.xlsx` | Dinner subset (meal_type_id = 3) |

### Pipeline stages
```
1. Load raw data          ← 3 Excel files from Google Drive
2. Join / merge           ← Candidate keys, union, melt, inner join
3. Data cleaning          ← Drop zeros, normalize Arabic text, date features
4. Word2Vec embeddings    ← Meal text → dense numerical vectors
5. Export                 ← Save per-meal CSV files to Drive
6. EDA                    ← Exploratory analysis on raw tables (informational)
```

>  **Run order matters.** Steps 1–5 are sequential and stateful. Do not skip or reorder cells.


###<u> To get access to data </u>:    
1 - Yacine probably shared with you a Drive with the folder *intellicanteen data* containing the data ; if not ask one of us to share it with you.


2 - Connect to your drive , in **Shared with me** you find the folder , **right click** on it.

3 - Then do **Organize > Add Shortcut > All locations > My Drive > Add**    

4 - Now you can execute directly the cell **below** to mount your google drive ,if google colab asks you access give it and connect ; **the path to the folder and the workspace is done directly (just don't change folder/files name)**



# Read the datasets

### 1.1 — Mount Google Drive




In [ ]:
 from google.colab import drive
 drive.mount('/content/drive')

Mounted at /content/drive


### 1.2 — Load Raw Data Files

We load three source Excel files that together describe the canteen system:

| Variable | File | Description |
|---|---|---|
| `menu_df` | `Menu_...xlsx` | Daily menus per restaurant, meal type, and date. Each row is one dish served at one meal. |
| `stats_day_df` | `Statistics_per_day_...xlsx` | Aggregate daily visitor counts per restaurant (breakfast / lunch / dinner in wide format). |
| `stats_client_type_df` | `Statistics_per_day_type_client_...xlsx` | Same daily counts broken down by client category (student, employee, etc.). |

>  All files must be present under `MyDrive/intellicanteen data/` before proceeding.


In [ ]:
from pathlib import Path
import pandas as pd

# Read the three project data files from Google Drive
DATA_DIR = Path("/content/drive/MyDrive/intellicanteen data")

menu_df = pd.read_excel("/content/drive/MyDrive/intellicanteen data/Copy of Copy of Menu_20260411_anonymized.xlsx")
stats_day_df = pd.read_excel("/content/drive/MyDrive/intellicanteen data/Copy of Copy of Statistics_per_day_20260410_anonymized.xlsx")
stats_client_type_df = pd.read_excel("/content/drive/MyDrive/intellicanteen data/Copy of Copy of Statistics_per_day_type_client_20260410_anonymized.xlsx")



print("Successfully read all 3 data files from Google Drive.")

Successfully read all 3 data files from Google Drive.


---
##  Phase 2 — Joining the Three Source Files

The three raw tables use different granularities and key schemes. This phase resolves those differences
step by step before producing a single analysis-ready table.

**Overall strategy:**
1. Drop irrelevant client-type columns from `stats_client_type_df`.
2. Collapse duplicate menu rows (same restaurant × date × meal type) into one comma-separated meal string.
3. Remove exact duplicates and identify candidate keys in each table.
4. Union `stats_day_df` and `stats_client_type_df` (the day-level stats table is a superset).
5. Melt the wide stats table into long format (one row per meal type per day).
6. Join the melted stats with the menu on `[meal_type_id, dou_code, date]`.
7. Drop administrative columns that carry no predictive value.
8. Persist the joined table to Drive for downstream notebooks.

The helper functions defined in the next two cells are reused across multiple steps.


### 2.1 — Utility Functions

**`explore_df(df)`** — prints a quick structural profile of any DataFrame: shape, dtypes,
missing value counts, duplicate row count, and per-column unique value samples.
Used below to audit all three raw tables before any transformation.


In [ ]:
import pandas as pd

def explore_df(df, max_unique=20):
    """
    Display useful information about a pandas DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
    max_unique : int
        Max number of unique values to display per column
    """

    print("\n===== BASIC INFO =====")
    print(f"Shape: {df.shape}")
    print(f"Rows : {df.shape[0]}")
    print(f"Cols : {df.shape[1]}")

    print("\n===== COLUMN TYPES =====")
    print(df.dtypes)

    print("\n===== MEMORY USAGE =====")
    print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

    print("\n===== MISSING VALUES =====")
    print(df.isna().sum())

    print("\n===== DUPLICATED ROWS =====")
    print(df.duplicated().sum())

    print("\n===== NUMERICAL SUMMARY =====")
    print(df.describe(include='number').T)

    print("\n===== CATEGORICAL SUMMARY =====")
    print(df.describe(include='object').T)

    print("\n===== UNIQUE VALUES =====")
    for col in df.columns:
        uniques = df[col].dropna().unique()
        n_unique = len(uniques)

        print(f"\nColumn: {col}")
        print(f"Unique count: {n_unique}")

        if n_unique <= max_unique:
            print(f"Values: {uniques}")
        else:
            print(f"First {max_unique} values: {uniques[:max_unique]}")

### 2.2 — Initial Exploration of All Three Raw Tables

All three DataFrames are collected into a single dictionary `dfs` for convenient iteration.
`explore_df` is then called on each to surface any immediate data quality issues
(unexpected nulls, wrong dtypes, hidden duplicates) before we touch the data.


In [ ]:
dfs = {
    "menu_df" : menu_df,
    "stats_day_df" : stats_day_df,
    "stats_client_type_df": stats_client_type_df
}

del menu_df , stats_day_df , stats_client_type_df

for name , df in dfs.items():
  print("=" * 25 , f"dataset : {name}", "=" * 25)
  explore_df(df)


========================= dataset : menu_df =========================

===== BASIC INFO =====
Shape: (570803, 7)
Rows : 570803
Cols : 7

===== COLUMN TYPES =====
meal            object
quantity        object
date            object
meal_type_id     int64
name            object
email            int64
id               int64
dtype: object

===== MEMORY USAGE =====
162.32016563415527 MB

===== MISSING VALUES =====
meal                0
quantity        35699
date                0
meal_type_id        0
name                0
email               0
id                  0
dtype: int64

===== DUPLICATED ROWS =====
4925

===== NUMERICAL SUMMARY =====
                 count        mean         std   min    25%    50%    75%  \
meal_type_id  570803.0    2.138692    0.763685   1.0    2.0    2.0    3.0   
email         570803.0  224.547445  126.768317  11.0  121.0  221.0  313.0   
id            570803.0   33.245629   18.662895   1.0   17.0   33.0   49.0   

                max  
meal_type_id    3.0  
em

### 2.3 — Key Discovery & Merge Helper Functions

Three functions are defined here and used in the pipeline steps below:

| Function | Purpose |
|---|---|
| `find_candidate_keys(df, max_cols)` | Brute-forces all minimal column combinations that uniquely identify every row. Avoids manual guessing of primary keys. |
| `merge_meals_by_key(df)` | Collapses multiple menu rows that share the same `(meal_type_id, date, id)` key into a single comma-separated `meal` string. Required because one meal slot can contain several dishes. |
| `union_and_count_conflicts(stats_day_df, stats_client_type_df)` | Performs a full union of the two stats tables (removing exact duplicates) and reports how many composite keys appear more than once with differing column values. |


In [ ]:
import itertools

def find_candidate_keys(df, max_cols=3):
    """
    Find minimal candidate keys of a dataframe.
    max_cols prevents combinatorial explosion.
    """

    n_rows = len(df)
    cols = list(df.columns)

    candidate_keys = []

    # test combinations
    for r in range(1, max_cols + 1):
        for comb in itertools.combinations(cols, r):

            # skip if superset of existing key
            if any(set(k).issubset(comb) for k in candidate_keys):
                continue

            # uniqueness test
            if df[list(comb)].drop_duplicates().shape[0] == n_rows:
                candidate_keys.append(comb)

    return candidate_keys

def merge_meals_by_key(df: pd.DataFrame) -> pd.DataFrame:
    """
    Merges rows with the same candidate key (meal_type_id, date, id) by concatenating
    their 'meal' values into a single comma-separated string.

    Args:
        df (pd.DataFrame): The original menu_df dataset.

    Returns:
        pd.DataFrame: A new, optimized dataframe with the merged rows.
    """
    # 1. Drop the 'quantity' column immediately to free up memory before grouping
    if 'quantity' in df.columns:
        df = df.drop(columns=['quantity'])

    # 2. Ensure 'meal' is strictly a string to avoid errors during joining
    df['meal'] = df['meal'].astype(str)

    # 3. Group by the candidate keys.
    # We include 'name' and 'email' in the groupby to retain them automatically.
    # Using as_index=False is faster and avoids the need for .reset_index() later.
    merged_df = df.groupby(
        ['meal_type_id', 'date', 'id', 'name', 'email'],
        as_index=False,
        dropna=False
    ).agg({
        'meal': ','.join  # Highly optimized C-level string joining in Pandas
    })

    return merged_df
import pandas as pd

def union_and_count_conflicts(stats_day_df: pd.DataFrame, stats_client_type_df: pd.DataFrame):
    """
    Performs a union of two dataframes without duplicates, prints its shape,
    and counts the number of key conflicts based on (id, create_date).

    Args:
        stats_day_df (pd.DataFrame): First dataframe
        stats_client_type_df (pd.DataFrame): Second dataframe

    Returns:
        tuple: (union_df, num_conflicts)
    """

    # 1. Union without duplicates
    # concat stacks them vertically, drop_duplicates removes completely identical rows
    union_df = pd.concat([stats_day_df, stats_client_type_df], ignore_index=True).drop_duplicates()

    # 2. Print the shape of the union
    print(f"Shape of the union DataFrame (without duplicates): {union_df.shape}")

    # 3. Count the conflicts
    # We group by the composite key and count the occurrences.
    # Since exact duplicates are gone, a size > 1 means the same key has differing values in other columns.
    key_counts = union_df.groupby(['resto_name','dou_code', 'create_date'], dropna=False).size()
    num_conflicts = (key_counts > 1).sum()

    print(f"Number of conflicts (same key, different values): {num_conflicts}")

    return union_df, num_conflicts

# Example of how to call the function:
# final_df, conflicts = union_and_count_conflicts(stats_day_df, stats_client_type_df)


### 2.4 — Instrumented Join Function

**`join_and_evaluate(df1, df2, left_on, right_on, how)`** wraps `pd.merge` with diagnostic output:
- Prints matched / unmatched row counts for both sides.
- Detects **row multiplication** (a 1-to-many join expanding the left table) and raises a warning,
  since unintended row explosion would inflate training data and skew predictions.
- Returns the clean merged DataFrame (indicator column dropped).


In [ ]:
from typing import Union, List

def join_and_evaluate(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    left_on: Union[str, List[str]],
    right_on: Union[str, List[str]],
    how: str = 'left'
) -> pd.DataFrame:
    """
    Joins two dataframes on 1, 2, or 3 columns and prints detailed matching statistics.

    Args:
        df1 (pd.DataFrame): The left dataframe.
        df2 (pd.DataFrame): The right dataframe.
        left_on (str or list): Column name(s) in the left dataframe to join on.
        right_on (str or list): Column name(s) in the right dataframe to join on.
        how (str): Type of join ('left', 'inner', 'outer', 'right'). Default is 'left'.

    Returns:
        pd.DataFrame: The successfully merged dataframe.
    """

    # Convert single strings to lists for consistent handling
    if isinstance(left_on, str): left_on = [left_on]
    if isinstance(right_on, str): right_on = [right_on]

    print("================ JOIN EVALUATION ================")
    print(f"Left DataFrame shape : {df1.shape}")
    print(f"Right DataFrame shape: {df2.shape}")
    print(f"Join Keys            : Left -> {left_on} | Right -> {right_on}")
    print(f"Join Type            : '{how}'\n")

    # Perform the merge
    # indicator=True creates a '_merge' column tracking the source of each row
    # suffixes resolves naming conflicts for columns that are NOT join keys
    merged_df = pd.merge(
        df1,
        df2,
        left_on=left_on,
        right_on=right_on,
        how=how,
        indicator=True,
        suffixes=('_df1', '_df2')
    )

    # Calculate matching statistics
    match_stats = merged_df['_merge'].value_counts()
    both_count = match_stats.get('both', 0)
    left_only_count = match_stats.get('left_only', 0)
    right_only_count = match_stats.get('right_only', 0)

    print("--- Matching Statistics ---")
    print(f"✅ Successfully matched in both : {both_count} rows")

    if how in ['left', 'outer']:
        print(f"❌ Unmatched (in df1 only)    : {left_only_count} rows")
    if how in ['right', 'outer']:
        print(f"❌ Unmatched (in df2 only)    : {right_only_count} rows")

    print(f"\nFinal Merged Shape: {merged_df.shape}")

    # --- Check for row multiplication (Data Explosion) ---
    # If a left join results in more rows than df1, it means the keys in df2 are NOT unique
    if how == 'left' and len(merged_df) > len(df1):
        extra_rows = len(merged_df) - len(df1)
        print(f"\n⚠️ WARNING: Row Multiplication Detected!")
        print(f"The merged dataframe has {extra_rows} MORE rows than the left dataframe.")
        print("This indicates a 1-to-many or many-to-many relationship. Ensure your join keys are unique in df2 if you expected a 1-to-1 match.")

    # Drop the temporary indicator column to keep the final dataframe clean
    merged_df = merged_df.drop(columns=['_merge'])
    print("=================================================")

    return merged_df

### 2.5 — Pipeline Execution (Steps 1 – 3)

A `skip_step` dictionary acts as a **run-control switch**: set a key to `1` to skip that step
(useful when the intermediate result was already saved to Drive and re-running the full pipeline
would be expensive).

| Step | Action | Rationale |
|---|---|---|
| **1** | Drop 12 client-type columns from `stats_client_type_df` | These breakdown columns (student/employee/doctor splits) are not needed for the regression target; removing them before key discovery avoids polluting the candidate key search. |
| **1.5** | Collapse menu rows → one row per `(meal_type_id, date, restaurant)` | The raw menu file has one row per dish; the stats tables have one row per meal session. Merging them without collapsing would create a many-to-one explosion. |
| **2** | Remove exact duplicates; print candidate keys for all three tables | Confirms the minimal key for each table so subsequent joins use the correct columns. |
| **3** | Drop `id` from both stats tables; drop `name` from menu | These administrative columns do not survive the join unambiguously and carry no predictive signal. |


In [ ]:
skip_step ={
    "1" : 0,
    "1.5" : 0,
    "2" : 0,
    "3" : 0,
    "4" : 0,
    "5" : 0,
    "6" : 0,
    "7" : 0,
    "8": 1
}
# 1 - remove some useless before checking candidate key
if not skip_step["1"] :
  dfs["stats_client_type_df"].drop(columns = ["dinner_para_medical","launch_para_medical","breakfast_para_medical","dinner_doctorant","launch_doctorant","breakfast_doctorant","dinner_employe","launch_employe","breakfast_employe","dinner_etudiant","launch_etudiant","breakfast_etudiant"]
                                 ,inplace=True)


# 1.5 : merge rows of menu_df having same candidate key
if not skip_step["1.5"] :
  print("="*10,"Entering step 1.5","="*10)
  dfs["menu_df"] = merge_meals_by_key(dfs["menu_df"])
  print("="*10,"leaving step 1.5","="*10)

  print(  dfs["menu_df"].head(10))
# 2 - remove duplicates and check for candidate keys
if not skip_step["2"] :
  for name , df in dfs.items():
    df.drop_duplicates()
    print(f"Duplicates of {name} removed .")
    print(f"Candidate keys for {name} are : {find_candidate_keys(df=df, max_cols=4)}")


# 3 - drop more useless columns after knowing candidate keys ,keep dou_code to merge with menu_df
if not skip_step["3"] :
  dfs["stats_client_type_df"].drop(columns=[ "id"]
                                  ,inplace=True)
  dfs["stats_day_df"].drop(columns=["id"]
                                  ,inplace=True)
  dfs["menu_df"].drop(columns=["name"],inplace=True)




========== Entering step 1.5 ==========
========== leaving step 1.5 ==========
   meal_type_id                 date  id               name  email  \
0             1  2005-02-02 00:00:00  12  ANON_d25e7b1824b7     81   
1             1  2015-02-18 00:00:00  24  ANON_27d112cfe108    161   
2             1  2021-10-17 00:00:00  27  ANON_6c7d50279296    171   
3             1  2023-01-26 00:00:00  40  ANON_b6378b028c28    253   
4             1  2023-02-18 00:00:00  12  ANON_d25e7b1824b7     81   
5             1  2023-02-20 00:00:00  22  ANON_3fb90f764317    152   
6             1  2023-02-24 00:00:00  25  ANON_9764962dbd29    162   
7             1  2023-10-14 00:00:00  40  ANON_b6378b028c28    253   
8             1  2023-10-29 00:00:00  21  ANON_ed27bbfdb259    151   
9             1  2023-10-30 00:00:00  21  ANON_ed27bbfdb259    151   

                                       meal  
0                                 قهوة,حليب  
1            قهوة بالحليب,بريوش,عصير,ياوورت  
2           

### 2.6 — Melt: Wide → Long Format (`expand_meals_to_rows`)

The stats tables store breakfast, lunch, and dinner visitor counts as **three separate columns**
on a single row. The menu table, by contrast, has one row per meal type.
To join them we must first unpivot the stats table:

```
Before melt (wide):                        After melt (long):
resto | date  | breakfast | lunch | dinner    resto | date  | meal_type_id | new_count
RU1   | 2024  |    120    |  200  |   95      RU1   | 2024  |      1       |   120
                                            RU1   | 2024  |      2       |   200
                                            RU1   | 2024  |      3       |    95
```

Meal type IDs: `1 = breakfast`, `2 = lunch`, `3 = dinner`.
This long format aligns with the `meal_type_id` column in `menu_df`, enabling the final join.


In [ ]:

def expand_meals_to_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms the wide meal dataframe into a long format.
    Splits the 'breakfast', 'launch', and 'dinner' columns into 3 separate rows
    per original row, assigning a meal_type_id and the corresponding count.
    """
    # 1. Define the columns we want to keep exactly as they are (identifiers)
    id_cols = ['create_date', 'resto_name', 'dou_code', 'id_pro']

    # 2. Define the columns we want to melt/unpivot
    meal_cols = ['breakfast', 'launch', 'dinner']
     # if it already exists in the dataframe.
    if 'count' in df.columns:
        df_for_melt = df.drop(columns=['count'])
    else:
        df_for_melt = df
    # 3. Perform the Melt operation
    # Notice we intentionally leave out the original 'count' column from id_vars
    # because it will be overwritten by the new specific meal count in value_name.
    long_df = pd.melt(
        df,
        id_vars=id_cols,
        value_vars=meal_cols,
        var_name='meal_name',   # temporary column to hold the string 'breakfast', etc.
        value_name='new_count'      # this becomes the new 'count' column for the specific meal
    )

    # 4. Map the string meal names to your requested numerical IDs
    meal_mapping = {
        'breakfast': 1,
        'launch': 2,
        'dinner': 3
    }
    long_df['meal_type_id'] = long_df['meal_name'].map(meal_mapping)

    # 5. Drop the temporary 'meal_name' string column as requested
    long_df = long_df.drop(columns=['meal_name'])

    # 6. (Optional) Sort to keep the 3 generated rows from the same original row together
    long_df = long_df.sort_values(by=id_cols + ['meal_type_id']).reset_index(drop=True)

    return long_df

### 2.7 — Pipeline Execution (Steps 4 – 8)

| Step | Action | Key Decision |
|---|---|---|
| **4** | Union `stats_day_df` ∪ `stats_client_type_df` | EDA showed that `stats_day_df` is a superset of `stats_client_type_df` on the composite key `(resto_name, dou_code, create_date)`. The union consolidates both without duplicating rows. |
| **5** | Melt union → long format via `expand_meals_to_rows` | Creates one row per (restaurant × date × meal type) — the granularity required to join with the menu. |
| **6** | Date normalization + inner join with `menu_df` | `menu_df.date` contains mixed-format timestamps (some with times, one batch with a typo `202-` instead of `2023-`). Both date columns are normalized to midnight before joining on `[meal_type_id, dou_code, date]`. An **inner join** is used: rows with no matching menu entry or no matching stats entry are discarded. |
| **7** | Drop post-join redundant columns (`id_pro`, `date`, `id`, `email`) | After the join these columns are either duplicated from the menu side or carry no modeling value. |
| **8** | Export `dfs['joined']` to Drive as `Merged_data.csv` | Persists the merged table so subsequent runs can skip steps 1–7 by setting their skip flags to `1`. |

>  **Join keys:** `meal_type_id` (meal session) + `dou_code` (restaurant code) + `create_date / date` (calendar day).


In [ ]:
### Step 4 : Union of stats_day_df and stats_client_type_df
## From this we learn that the true key of stats_client_type_df and stats_day_df is [resto_name, dou_code, create_date] and that the first df is a superset of the second
if not skip_step["4"]:
  dfs["union1"] , conflicts= union_and_count_conflicts(stats_day_df = dfs['stats_day_df'] , stats_client_type_df = dfs['stats_client_type_df'])
  print(f"Original shape of stats_day_df : {dfs["stats_day_df"].shape}\nOriginal shape of stats_client_type_df : {dfs["stats_client_type_df"].shape}\n")
  print(f"Number of conflicts after union (stats_day_df U stats_client_type_df) is {conflicts} \n Shape of Union : {dfs["union1"].shape}")

# Print nicely union
# print(dfs["union1"].columns.tolist())

##step 5 Melt union1 df to make divide between breakfasts , launch and dinners
if not skip_step["5"]:
  dfs["expanded_union"] = expand_meals_to_rows(dfs['union1'])

##step 6 Join expanded_union with
if not skip_step["6"]:
  # The format='mixed' tells pandas not to crash if some rows have times and some don't
  dfs["menu_df"]['date'] = pd.to_datetime(
      dfs["menu_df"]['date'],
      format='mixed',
      errors='coerce'  # We keep 'coerce' to handle any other completely broken text
  )


  dfs["menu_df"]['date'] = dfs["menu_df"]['date'].astype(str).str.replace('202-', '2023-')
  # Convert the 'date' column in the right dataframe to datetime
  dfs["menu_df"]['date'] = pd.to_datetime(dfs["menu_df"]['date'])
  # This forces both columns to midnight (00:00:00) so they match perfectly on the day
  dfs["menu_df"]['date'] = dfs["menu_df"]['date'].dt.normalize()
  dfs["expanded_union"]['create_date'] = dfs["expanded_union"]['create_date'].dt.normalize()


    # Do the same for the left dataframe just to be 100% safe
  dfs["expanded_union"]['create_date'] = pd.to_datetime(dfs["expanded_union"]['create_date'])
# 'meal_type_id' is the same, but 'email' and 'date' differ slightly in naming
  dfs["joined"] = join_and_evaluate(
    dfs["expanded_union"],
    dfs["menu_df"],
    left_on=['meal_type_id', 'dou_code', 'create_date'],
    right_on=['meal_type_id', 'email', 'date'],
    how='inner'
  )

# 7 again drop unvaluable columns for train after final merge
if not skip_step["7"]:

  print(dfs["joined"].columns.tolist())
  dfs["joined"].drop(columns=['id_pro','date','id','email'],inplace=True)

if not skip_step["8"]:
  print("step 8 entered")
  dfs["joined"].to_csv(
      "/content/drive/MyDrive/Merged_data.csv",
      index=False
  )

Shape of the union DataFrame (without duplicates): (250249, 8)
Number of conflicts (same key, different values): 0
Original shape of stats_day_df : (249837, 8)
Original shape of stats_client_type_df : (250249, 8)

Number of conflicts after union (stats_day_df U stats_client_type_df) is 0 
 Shape of Union : (250249, 8)
================ JOIN EVALUATION ================
Left DataFrame shape : (750747, 6)
Right DataFrame shape: (115566, 5)
Join Keys            : Left -> ['meal_type_id', 'dou_code', 'create_date'] | Right -> ['meal_type_id', 'email', 'date']
Join Type            : 'inner'

--- Matching Statistics ---
✅ Successfully matched in both : 670882 rows

Final Merged Shape: (670882, 11)
['create_date', 'resto_name', 'dou_code', 'id_pro', 'new_count', 'meal_type_id', 'date', 'id', 'email', 'meal']


### 2.8 — Verify Joined Table Schema

Print the column list of `dfs['joined']` to confirm that the join produced the expected schema
before moving into cleaning. Expected columns at this point:
`create_date, resto_name, dou_code, new_count, meal_type_id, meal`.


In [ ]:
print(dfs["joined"].columns.tolist())


['create_date', 'resto_name', 'dou_code', 'new_count', 'meal_type_id', 'meal']


### 2.9 — Ensure Output Directory Exists

Creates the Drive output folder if it does not already exist. Safe to run even if the folder is present.


In [ ]:
!mkdir -p "/content/drive/MyDrive/intellicanteen data"

---
##  Phase 3 — Data Cleaning & Feature Engineering

The joined table is now processed into a model-ready DataFrame. This phase covers:

1. **Zero-count removal** — days with `new_count == 0` indicate closed restaurants or data gaps; they are excluded to prevent the model from learning a spurious "zero pattern".
2. **Arabic text normalization** — meal descriptions contain dialect spelling variants, diacritics, and mixed-script characters. A custom cleaner standardizes them so that `Word2Vec` treats semantically identical meals as the same token.
3. **Date feature extraction** — `create_date` is decomposed into `year`, `month`, `day`, and seven binary `is_<DayName>` columns (one-hot encoding of the day of the week). These are the primary calendar features used by the downstream models.


### 3.1 — Extract Working DataFrame

Pull `dfs['joined']` into a standalone variable `df` for the cleaning pipeline.
Printing the column list here serves as a final schema check before transformation.


In [ ]:
df = dfs['joined']
print("The dataset columns : ",df.columns.tolist())

The dataset columns :  ['create_date', 'resto_name', 'dou_code', 'new_count', 'meal_type_id', 'meal']


### 3.2 — Arabic Text Normalization & Full Preprocessing Function

**`clean_arabic_text(text)`** applies six normalization steps to each meal string:

| Step | Operation | Example |
|---|---|---|
| 1 | Lowercase Latin/French characters | `Poulet` → `poulet` |
| 2 | Strip Arabic diacritics (Tashkeel) | `كُسْكُس` → `كسكس` |
| 3 | Normalize Alef variants | `أ / إ / آ / ا` → `ا` |
| 4 | Normalize Teh Marbuta & Alef Maksura | `ة → ه`, `ى → ي` |
| 5 | Remove Tatweel and punctuation | `كـسـكـس` → `كسكس` |
| 6 | Collapse extra whitespace | `couscous  poulet` → `couscous poulet` |

**`preprocess_training_data(df)`** orchestrates the full cleaning pipeline:
- Drops rows where `new_count == 0`.
- Applies `clean_arabic_text` to the `meal` column → new column `meal_cleaned`.
- Parses `create_date` as datetime, extracts `year`, `month`, `day`.
- One-hot encodes the day of the week into 7 binary columns (`is_Monday` … `is_Sunday`), ensuring all 7 columns exist even if some days are absent from the data.

> The function is applied to `df` in-place at the bottom of this cell.


In [ ]:
import pandas as pd
import re

def clean_arabic_text(text):
    """
    Cleans and normalizes Arabic text (specifically suited for Algerian dialect).
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # 1. Lowercase any Latin/French characters mixed in the text
    text = text.lower()

    # 2. Remove Arabic diacritics (Tashkeel)
    tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(tashkeel, '', text)

    # 3. Normalize Arabic letters (standardize Alef, Teh Marbuta, etc.)
    text = re.sub(r'[إأآا]', 'ا', text) # Normalize Alef
    text = re.sub(r'ة', 'ه', text)      # Convert Teh Marbuta to Heh (common in dialect)
    text = re.sub(r'ى', 'ي', text)      # Convert Alef Maksura to Yaa
    text = re.sub(r'ـ', '', text)       # Remove Tatweel (stretching character)

    # 4. Remove punctuation, special characters, and numbers (optional but recommended for pure NLP)
    # This regex keeps only Arabic letters, Latin letters, and spaces
    text = re.sub(r'[^\u0600-\u06FFa-zA-Z\s]', ' ', text)

    # 5. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def preprocess_training_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies the full preprocessing pipeline for the regression task.
    """
    print(f"Original shape: {df.shape}")

    # ==========================================
    # TASK 1: Drop rows where new_count == 0
    # ==========================================
    df = df[df['new_count'] != 0].copy()
    print(f"Shape after dropping 0s: {df.shape}")

    # ==========================================
    # TASK 2: Clean and Normalize 'meal' column
    # ==========================================
    df['meal_cleaned'] = df['meal'].apply(clean_arabic_text)

    # ==========================================
    # TASK 3: Date Feature Engineering
    # ==========================================
    # Ensure 'create_date' is a datetime object
    df['create_date'] = pd.to_datetime(df['create_date'])

    # Extract year, month, day
    df['year'] = df['create_date'].dt.year
    df['month'] = df['create_date'].dt.month
    df['day'] = df['create_date'].dt.day

    # Create the One-Hot Encoding for the days of the week
    # .dt.day_name() returns strings like 'Monday', 'Tuesday'
    # pd.get_dummies converts them into True/False (or 1/0) columns
    day_dummies = pd.get_dummies(df['create_date'].dt.day_name(), prefix='is').astype(int)

    # Ensure all 7 days exist even if some are missing in the dataset
    all_days = ['is_Monday', 'is_Tuesday', 'is_Wednesday', 'is_Thursday', 'is_Friday', 'is_Saturday', 'is_Sunday']
    for day in all_days:
        if day not in day_dummies.columns:
            day_dummies[day] = 0

    # Concatenate the new day columns to the main dataframe
    df = pd.concat([df, day_dummies[all_days]], axis=1)

    return df

# Example Usage:
df = preprocess_training_data(df)
print(df.head())

Original shape: (670882, 6)
Shape after dropping 0s: (576421, 6)
   create_date         resto_name  dou_code  new_count  meal_type_id  \
1   2023-12-12  ANON_294c1f5ad3ee       141          2             2   
4   2023-12-12  ANON_7adf4378a38c        52          1             2   
7   2023-12-12  ANON_a7c48f9a2252       121          1             2   
10  2023-12-12  ANON_b6d4d095157f        92         65             2   
13  2023-12-12  ANON_beae590631b8        41          6             2   

                                                 meal  \
1                أرز بالفرن,جبن,سلطة متنوعة,تحلية,خبز   
4              سلاطة متنوعة,عدس,بيض مسلوق,ياؤورت معطر   
7              سلاطة خس,عدس,باتي,ياؤورت معطر,خبز محسن   
10  خبز,سلاطة متنوعة,معكرونة بالصلصة,كاشير,جبن,فاك...   
13                 خبز,لوبيا,جبن,سلاطة ,مشروبات غازية   

                                         meal_cleaned  year  month  day  \
1                ارز بالفرن جبن سلطه متنوعه تحليه خبز  2023     12   12   
4        

---
##  Phase 4 — Meal Embeddings with Word2Vec

Machine learning models require purely numerical input. The `meal_cleaned` column is free text.
Rather than one-hot encoding thousands of individual dish names (sparse and brittle), we use
**Word2Vec** to project each meal into a dense vector space trained on our own menu corpus.

### Why train from scratch?
Pre-trained models (e.g., Google's 300-d vectors) know nothing about Algerian canteen food.
Training on our own data ensures that semantically similar meals (e.g., `couscous poulet` and `كسكس دجاج`)
end up close in the embedding space — regardless of what any general-purpose model would say.

### Vector size heuristic
A common starting point for embedding dimensionality is $\\text{vector\\_size} \\approx V^{1/4}$,
where $V$ is the vocabulary size. This keeps the embedding compact while retaining enough capacity
to represent the menu diversity. In practice we round to `12` dimensions.


 # **Training Word2Vec model on meal column :**
  Training a model strictly on our dataset, it will create a highly specialized mathematical space where it only understands the present menu.

If meal is "couscous poulet", the model will learn that "couscous" and "poulet" often appear together, placing them closer in the vector space, totally ignoring the rest of the world's language.

The best tool for this is `Word2Vec` using the `gensim` library.
Machine Learning engineers at Google often use a simple heuristic to calculate a starting vector size based on the total number of unique words in your vocabulary ($V$):$$Vector\_Size \approx \sqrt[4]{V}$$

### 4.1 — Install Gensim

The `gensim` library provides the `Word2Vec` implementation. The `-q` flag suppresses verbose install output.


In [ ]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 69.1 MB/s eta 0:00:00


### 4.2 — Word2Vec Training & Embedding Function

**`train_and_apply_food_word2vec(df, text_col, vector_size)`** performs four actions:

1. **Tokenize** — splits each cleaned meal string into a list of words (e.g., `['كسكس', 'دجاج']`).
2. **Train** — fits a `Word2Vec` model on all tokenized meals with a context window of 3 and `min_count=1` (no word is ignored, even rare dishes).
3. **Average** — for each meal, retrieves the embedding vector of every word and averages them into a single fixed-length vector representing the whole meal.
4. **Append** — adds `vector_size` new columns (`meal_w2v_1` … `meal_w2v_N`) to the DataFrame and drops the `meal_cleaned` text column.

The result is a fully numerical DataFrame ready for tree-based model training.


In [ ]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

def train_and_apply_food_word2vec(df: pd.DataFrame, text_col: str = 'meal_cleaned', vector_size: int = 15) -> pd.DataFrame:
    """
    Trains a custom Word2Vec model from scratch on the meal column,
    averages the word vectors for each meal, and adds them as new columns.
    """
    print(f"Starting custom Word2Vec embedding for {len(df)} rows...")

    # 1. Tokenize the text (split sentences into lists of words)
    # Example: "couscous poulet" -> ["couscous", "poulet"]
    # We drop empty strings to avoid errors
    sentences = df[text_col].fillna("").str.split().tolist()

    # 2. Train the Word2Vec model FROM SCRATCH
    # vector_size: How many columns we want in the end (e.g., 15)
    # window: How many words to look at around each word (small is good for short meal names)
    # min_count: 1 ensures we don't ignore rare meals
    # workers: Uses multi-threading to train faster
    print(f"Training Word2Vec model on the food space (vector_size={vector_size})...")
    model = Word2Vec(sentences=sentences, vector_size=vector_size, window=3, min_count=1, workers=4)

    # 3. Create a function to average the vectors of a meal
    # If a meal has 3 words, Word2Vec gives 3 vectors. We average them into 1 vector representing the whole meal.
    def get_meal_vector(words_list):
        # Filter out words that somehow didn't make it into the vocabulary
        valid_words = [word for word in words_list if word in model.wv.key_to_index]

        if len(valid_words) == 0:
            # If the meal is completely empty or unrecognized, return a vector of zeros
            return np.zeros(vector_size)
        else:
            # Get the vectors for all valid words and average them along the column axis
            word_vectors = np.array([model.wv[word] for word in valid_words])
            return word_vectors.mean(axis=0)

    # 4. Apply the function to all rows to get our final embeddings
    print("Converting meals to vectors...")
    meal_embeddings = np.array([get_meal_vector(words) for words in sentences])

    # 5. Add the new embedding columns to the DataFrame
    embed_cols = [f'meal_w2v_{i+1}' for i in range(vector_size)]
    embeddings_df = pd.DataFrame(meal_embeddings, columns=embed_cols, index=df.index)

    # Concatenate the new columns with the original dataframe
    final_df = pd.concat([df, embeddings_df], axis=1)

    # Drop the original text column since the machine learning model only wants numbers
    final_df = final_df.drop(columns=[text_col])

    print("✅ Custom Food Space Word2Vec complete!")
    return final_df



### 4.3 — Compute Vocabulary Size $V$

Before training, we need to know how many unique dish names exist in the entire corpus.
`get_total_unique_meals` tokenizes every row by comma (meals are comma-separated),
strips whitespace, and collects all tokens into a set (automatic deduplication).
The resulting count $V$ is used in the next cell to determine the embedding dimension.


In [ ]:
import pandas as pd

def get_total_unique_meals(df: pd.DataFrame, text_col: str = 'meal') -> int:
    """
    Scans the entire dataset, extracts all comma-separated meals,
    and returns a single integer representing the total unique vocabulary.
    """
    # A Python 'set' automatically prevents duplicates from being added
    global_unique_meals = set()

    # Drop missing values to prevent errors, and iterate over the text
    for meal_string in df[text_col].dropna():

        # Split by comma, clean spaces, and lowercase
        cleaned_meals = [m.strip().lower() for m in str(meal_string).split(',') if m.strip()]

        # .update() adds multiple items to the set at once (ignoring duplicates)
        global_unique_meals.update(cleaned_meals)

    # Return just the final integer count
    return len(global_unique_meals)

V = get_total_unique_meals(df, text_col='meal')
print(f"Total unique meals in dataset: {V}")

Total unique meals in dataset: 17196


### 4.4 — Train Word2Vec and Embed the Meal Column

The embedding dimension is computed as $V^{1/4}$ (rounded). We then call `train_and_apply_food_word2vec`
to train the model and append the `meal_w2v_1 … meal_w2v_12` columns to `df`.
After this cell, `df` contains no raw text — only numerical features.


In [ ]:
from math import sqrt
vector_size=sqrt(sqrt(V))
print("Vector size of word2vec : ",vector_size)
df = train_and_apply_food_word2vec(df, vector_size=12)

Vector size of word2vec :  11.451354493782997
Starting custom Word2Vec embedding for 576421 rows...
Training Word2Vec model on the food space (vector_size=12)...
Converting meals to vectors...
✅ Custom Food Space Word2Vec complete!


### 4.5 — Verify Final Column Schema

Print the full column list after embedding to confirm the W2V columns were appended correctly
and no unexpected columns remain.


In [ ]:
print("The dataset columns : ",df.columns.tolist())

The dataset columns :  ['create_date', 'resto_name', 'dou_code', 'new_count', 'meal_type_id', 'meal', 'year', 'month', 'day', 'is_Monday', 'is_Tuesday', 'is_Wednesday', 'is_Thursday', 'is_Friday', 'is_Saturday', 'is_Sunday', 'meal_w2v_1', 'meal_w2v_2', 'meal_w2v_3', 'meal_w2v_4', 'meal_w2v_5', 'meal_w2v_6', 'meal_w2v_7', 'meal_w2v_8', 'meal_w2v_9', 'meal_w2v_10', 'meal_w2v_11', 'meal_w2v_12']


---
##  Phase 5 — Export Processed Dataset

### 5.1 — Save Master Dataset

The fully preprocessed DataFrame is written to Drive as `dataset.xlsx`.
This file contains **all meal types** and serves as the master record.
The `create_date` column is intentionally retained (commented drop)
so that downstream notebooks can re-derive temporal features if needed.


In [ ]:

#df.drop(columns=["create_date"],inplace=True)
print(df.head())
df.to_csv(
      "/content/drive/MyDrive/intellicanteen data/dataset.xlsx",
      index=False
)

   create_date         resto_name  dou_code  new_count  meal_type_id  \
1   2023-12-12  ANON_294c1f5ad3ee       141          2             2   
4   2023-12-12  ANON_7adf4378a38c        52          1             2   
7   2023-12-12  ANON_a7c48f9a2252       121          1             2   
10  2023-12-12  ANON_b6d4d095157f        92         65             2   
13  2023-12-12  ANON_beae590631b8        41          6             2   

                                                 meal  year  month  day  \
1                أرز بالفرن,جبن,سلطة متنوعة,تحلية,خبز  2023     12   12   
4              سلاطة متنوعة,عدس,بيض مسلوق,ياؤورت معطر  2023     12   12   
7              سلاطة خس,عدس,باتي,ياؤورت معطر,خبز محسن  2023     12   12   
10  خبز,سلاطة متنوعة,معكرونة بالصلصة,كاشير,جبن,فاك...  2023     12   12   
13                 خبز,لوبيا,جبن,سلاطة ,مشروبات غازية  2023     12   12   

    is_Monday  ...  meal_w2v_3  meal_w2v_4  meal_w2v_5  meal_w2v_6  \
1           0  ...    1.668460    0.356258   -

### 5.2 — Split by Meal Type

The master dataset is filtered into three separate DataFrames using `meal_type_id`:

| Variable | `meal_type_id` | Used by |
|---|---|---|
| `df_breakfast` | 1 | `breakfast_pipeline_final.ipynb` |
| `df_lunch` | 2 | *(future lunch model)* |
| `df_dinner` | 3 | `dinner_pipeline_final.ipynb` |


In [ ]:
df_breakfast = df[df['meal_type_id'] == 1]
df_lunch = df[df['meal_type_id'] == 2]
df_dinner = df[df['meal_type_id'] == 3]
datasets = {"breakfast_data":df_breakfast,
            "lunch_data":df_lunch,
            "dinner_data":df_dinner}


### 5.3 — Export Per-Meal Files

Each subset is written to Drive as a separate `.xlsx` file.
The `meal_type_id` column is dropped at export time since it is constant within each file
and would add no information to the downstream models.

> ✅ **After this cell completes, the preprocessing pipeline is finished.**
The three exported files are the direct inputs to the modeling notebooks.


In [ ]:
for name, dataset in datasets.items():
  dataset.drop(columns=["meal_type_id"])
  dataset.to_csv(
      f"/content/drive/MyDrive/intellicanteen data/{name}.xlsx",
      index=False
  )

---
##  Phase 6 — Exploratory Data Analysis (EDA)

The following cells perform exploratory analysis on the **raw source tables**
(before joining and cleaning). This phase is **informational** — none of these cells
modify `df` or produce any exported file. Their purpose is to:

- Understand the structure and cardinality of each raw table.
- Audit data quality (missing values, duplicates, type inconsistencies).
- Verify key assumptions (e.g., whether `resto_name` is a reliable unique identifier).
- Inform decisions made in the joining and cleaning phases above.

>  These EDA cells can be run independently at any time on the original `dfs` dict without affecting the pipeline.


### columns, number of unique values per column, and total rows for each dataset

### 6.1 — Column Cardinality Summary

`print_unique_counts` iterates over all columns of each raw table and reports the number of distinct values.
High-cardinality columns (many unique values) are good candidates for keys or embeddings;
low-cardinality columns (few unique values) often represent categorical groupings suitable for one-hot encoding.


In [ ]:
import pandas as pd

# EDA Step 1: columns, number of unique values per column, and total rows for each dataset

def print_unique_counts(df, name):
    print(f"\n{name} - unique values per column")
    print(f"Total rows: {len(df)}")
    print("-" * 60)
    for col in df.columns:
        print(f"{col}: {df[col].nunique(dropna=False)}")

print_unique_counts(dfs["menu_df"], "menu_df")
print_unique_counts(dfs["stats_day_df"], "stats_day_df")
print_unique_counts(dfs["stats_client_type_df"], "stats_client_type_df")


menu_df - unique values per column
Total rows: 115566
------------------------------------------------------------
meal_type_id: 3
date: 841
id: 66
email: 66
meal: 69386

stats_day_df - unique values per column
Total rows: 249837
------------------------------------------------------------
create_date: 829
resto_name: 508
dou_code: 67
id_pro: 429
count: 3672
breakfast: 942
launch: 2346
dinner: 1923

stats_client_type_df - unique values per column
Total rows: 250249
------------------------------------------------------------
create_date: 830
resto_name: 508
dou_code: 67
id_pro: 429
count: 3673
breakfast: 942
launch: 2346
dinner: 1923


### 6.2 — Identity Consistency Check in `menu_df`

Counts the number of unique `(id, email)` pairs to verify that these two columns together form a reliable person identifier.
If the count is much smaller than the number of rows, the table is in a one-to-many structure (one person → many meal entries).


In [ ]:
# Number of unique combinations of id, email, and name in menu_df
unique_combinations = dfs["menu_df"].drop_duplicates(subset=["id", "email"]).shape[0]
print("Unique combinations of id, email:", unique_combinations)

Unique combinations of id, email: 66


### unique identifier in menu_df
in the dataset menu_df,
since there are only 66 unique combinations out of hundreds of thousands of rows, these three columns together essentially act as a single unique identifier for a person.

### 6.3 — Data Quality Audit

`data_quality_audit` produces a structured report per table:
- **Column dtypes** — catches mixed-type columns (e.g., dates stored as strings).
- **Missing value report** — ranked by frequency, so the most critical gaps are visible at a glance.
- **Exact duplicate count** — a non-zero count here would imply data ingestion issues.


In [ ]:
# EDA Step 2: Data Quality Audit (run this step only)

def data_quality_audit(df, name):
    print(f"\n{name} - Data Quality Audit")
    print("=" * 70)

    print(f"Rows: {len(df)} | Columns: {df.shape[1]}")

    print("\nColumn dtypes:")
    print(df.dtypes)

    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)
    missing_summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct
    }).sort_values("missing_count", ascending=False)

    print("\nMissing values (top 20 columns):")
    print(missing_summary.head(20))

    dup_rows = df.duplicated().sum()
    print(f"\nExact duplicate rows: {dup_rows} ({(dup_rows / len(df) * 100):.2f}%)")


data_quality_audit(dfs["menu_df"], "menu_df")
data_quality_audit(dfs["stats_day_df"], "stats_day_df")
data_quality_audit(dfs["stats_client_type_df"], "stats_client_type_df")


menu_df - Data Quality Audit
Rows: 115566 | Columns: 5

Column dtypes:
meal_type_id             int64
date            datetime64[ns]
id                       int64
email                    int64
meal                    object
dtype: object

Missing values (top 20 columns):
              missing_count  missing_pct
date                     64         0.06
meal_type_id              0         0.00
id                        0         0.00
email                     0         0.00
meal                      0         0.00

Exact duplicate rows: 2 (0.00%)

stats_day_df - Data Quality Audit
Rows: 249837 | Columns: 8

Column dtypes:
create_date    datetime64[ns]
resto_name             object
dou_code                int64
id_pro                float64
count                   int64
breakfast               int64
launch                  int64
dinner                  int64
dtype: object

Missing values (top 20 columns):
             missing_count  missing_pct
id_pro                4500          1.8
c

### 6.4 — Granularity & Cross-Table Key Overlap *(archived)*

This cell contains an extended key-uniqueness and cross-dataset overlap analysis.
It is **commented out** because the relevant findings were already incorporated into the pipeline design.
Uncomment to re-run the full diagnostic if the source data changes.


In [ ]:
# # EDA Step 3: Granularity and Key Consistency (run this step only)

# def check_key_uniqueness(df, cols, df_name):
#     dup_count = df.duplicated(subset=cols).sum()
#     pct = dup_count / len(df) * 100
#     print(f"{df_name} | Key {cols} -> duplicates: {dup_count} ({pct:.2f}%)")


# def check_one_to_one(df, left_col, right_col, df_name):
#     left_to_right_max = df.groupby(left_col)[right_col].nunique(dropna=False).max()
#     right_to_left_max = df.groupby(right_col)[left_col].nunique(dropna=False).max()
#     is_one_to_one = (left_to_right_max == 1) and (right_to_left_max == 1)

#     print(
#         f"{df_name} | {left_col} <-> {right_col}: "
#         f"one_to_one={is_one_to_one}, "
#         f"max_unique_{right_col}_per_{left_col}={left_to_right_max}, "
#         f"max_unique_{left_col}_per_{right_col}={right_to_left_max}"
#     )


# print("\nmenu_df - Granularity/Key Consistency")
# print("=" * 70)
# print("Candidate grain checks:")
# check_key_uniqueness(menu_df, ["date", "id", "meal", "meal_type_id"], "menu_df")
# check_key_uniqueness(menu_df, ["date", "id", "meal_type_id"], "menu_df")
# check_key_uniqueness(menu_df, ["date", "id"], "menu_df")

# print("\nIdentity consistency checks:")
# check_one_to_one(menu_df, "id", "email", "menu_df")
# check_one_to_one(menu_df, "id", "name", "menu_df")
# check_one_to_one(menu_df, "email", "name", "menu_df")


# print("\nstats_day_df - Granularity/Key Consistency")
# print("=" * 70)
# print("Candidate grain checks:")
# check_key_uniqueness(stats_day_df, ["create_date", "resto_name", "dou_code"], "stats_day_df")
# check_key_uniqueness(stats_day_df, ["create_date", "id"], "stats_day_df")
# check_key_uniqueness(stats_day_df, ["create_date", "id_pro"], "stats_day_df")


# print("\nstats_client_type_df - Granularity/Key Consistency")
# print("=" * 70)
# print("Candidate grain checks:")
# check_key_uniqueness(stats_client_type_df, ["create_date", "resto_name", "dou_code"], "stats_client_type_df")
# check_key_uniqueness(stats_client_type_df, ["create_date", "id"], "stats_client_type_df")
# check_key_uniqueness(stats_client_type_df, ["create_date", "id_pro"], "stats_client_type_df")


# print("\nCross-dataset key overlap (quick sanity)")
# print("=" * 70)
# common_daily_keys = set(stats_day_df[["create_date", "resto_name", "dou_code"]].drop_duplicates().apply(tuple, axis=1))
# common_client_keys = set(stats_client_type_df[["create_date", "resto_name", "dou_code"]].drop_duplicates().apply(tuple, axis=1))

# intersection_count = len(common_daily_keys & common_client_keys)
# only_day = len(common_daily_keys - common_client_keys)
# only_client = len(common_client_keys - common_daily_keys)

# print(f"Shared keys: {intersection_count}")
# print(f"Only in stats_day_df: {only_day}")
# print(f"Only in stats_client_type_df: {only_client}")

### stats_day_df
Check whether the same resto_name always implies the same dou_code, id, and id_pro in stats_day_df

### 6.5 — Restaurant Name Consistency Check

Verifies whether `resto_name` uniquely determines `dou_code` and `id_pro`.
For each target column, we count how many `resto_name` values map to more than one distinct value.
A consistent mapping (max unique = 1) would allow `resto_name` to serve as a standalone key;
any violation means we must use a composite key for reliable identification.


In [ ]:
# Check whether the same resto_name always implies the same dou_code, id, and id_pro in stats_day_df
key_columns = ["dou_code", "id_pro"]

print("stats_day_df - resto_name consistency check")
print("=" * 70)

summary = []
for column in key_columns:
    unique_per_resto = dfs["stats_day_df"].groupby("resto_name")[column].nunique(dropna=False)
    consistent = (unique_per_resto == 1).all()
    violating_resto_count = (unique_per_resto > 1).sum()
    max_unique = unique_per_resto.max()

    print(f"resto_name -> {column}: consistent={consistent}, violating_resto_name_count={violating_resto_count}, max_unique_{column}_per_resto_name={max_unique}")

    summary.append({
        "column": column,
        "consistent": consistent,
        "violating_resto_name_count": int(violating_resto_count),
        "max_unique_values_per_resto_name": int(max_unique),
    })

summary_df = pd.DataFrame(summary)
print("\nSummary:")
print(summary_df)

violations = {
    column: unique_per_resto[unique_per_resto > 1].index.tolist()
    for column in key_columns
}

print("\nSample violating resto_name values (up to 10 per column):")
for column, values in violations.items():
    print(f"{column}: {values[:10]}")

stats_day_df - resto_name consistency check
resto_name -> dou_code: consistent=False, violating_resto_name_count=13, max_unique_dou_code_per_resto_name=6
resto_name -> id_pro: consistent=False, violating_resto_name_count=13, max_unique_id_pro_per_resto_name=6

Summary:
     column  consistent  violating_resto_name_count  \
0  dou_code       False                          13   
1    id_pro       False                          13   

   max_unique_values_per_resto_name  
0                                 6  
1                                 6  

Sample violating resto_name values (up to 10 per column):
dou_code: ['ANON_0355b885c8b0', 'ANON_153e1dcd715a', 'ANON_288b1df54a1c', 'ANON_2ea10f4e8507', 'ANON_7b0e9fe69b3b', 'ANON_8afcbd993477', 'ANON_8f2b1cc296f2', 'ANON_af3d9da1b6cf', 'ANON_c21ffece397d', 'ANON_ca3931ab4c3e']
id_pro: ['ANON_0355b885c8b0', 'ANON_153e1dcd715a', 'ANON_288b1df54a1c', 'ANON_2ea10f4e8507', 'ANON_7b0e9fe69b3b', 'ANON_8afcbd993477', 'ANON_8f2b1cc296f2', 'ANON_af3d9da1

## Interpretation of the results

The identity columns in `menu_df` are perfectly consistent: `id`, `email`, and `name` form a one-to-one relationship, so they can safely be treated as a single person identifier.

For `stats_day_df`, however, the same `resto_name` does **not** always imply the same `dou_code`, `id`, or `id_pro`. In all three cases, 13 restaurant names map to more than one value, and each of those restaurants can be associated with up to 6 different values. This means `resto_name` is not a unique key by itself.

So, `resto_name` should not be used as a standalone identifier for those columns. If we need a stable key, we should rely on a stronger combination such as `create_date + resto_name + dou_code` or investigate whether the variation comes from duplicated naming across locations, data quality issues, or legitimate multi-branch restaurants.

### 6.6 — Preview Raw Menu Data

Display the first 20 rows of `menu_df` to inspect the raw structure of meal descriptions,
including formatting quirks, mixed Arabic/French text, and date formats.


In [ ]:
# print the first 20 rows of menu
print(f"menu_df - first 20 rows:")
print(dfs["menu_df"].head(20))

menu_df - first 20 rows:
    meal_type_id       date  id  email  \
0              1 2005-02-02  12     81   
1              1 2015-02-18  24    161   
2              1 2021-10-17  27    171   
3              1 2023-01-26  40    253   
4              1 2023-02-18  12     81   
5              1 2023-02-20  22    152   
6              1 2023-02-24  25    162   
7              1 2023-10-14  40    253   
8              1 2023-10-29  21    151   
9              1 2023-10-30  21    151   
10             1 2023-10-31  21    151   
11             1 2023-11-01  21    151   
12             1 2023-11-02  21    151   
13             1 2023-11-03  40    253   
14             1 2023-11-03  52    341   
15             1 2023-11-04  38    251   
16             1 2023-11-04  52    341   
17             1 2023-11-04  60    421   
18             1 2023-11-05   1     11   
19             1 2023-11-05   3     31   

                                        meal  
0                                  قهوة,حليب 

### 6.7 — Sample of Unique Meal Names

Print 100 raw unique values from the `meal` column. This helps identify the scale and variety of the vocabulary,
common spelling variants (e.g., `شكولاطة` vs `شيكولاطة`),
and mixed-script entries that the text cleaner must handle.


In [ ]:
print("menu_df - 100 unique values from the column meal:")
for val in dfs["menu_df"]["meal"].unique()[:100]:
    print(val)

menu_df - 100 unique values from the column meal:
قهوة,حليب
قهوة بالحليب,بريوش,عصير,ياوورت
حليب بالقهوة,مربى,بيض,خبز
حليب,قهوة,هلاليات,زبدة,خبز محسن
قهوة,حليب,كروكي
قهوة بالحليب,هلاليات
قهوة بالحليب,بريوش,تفاح
حليب,قهوة,زبدة,هلاليات,خبز محسن
حليب بالقهوة-خبز بالمربى,كاس الحليب,مربى
حليب بالقهوة-حبز بالمربى
حليب بالقهوة -خبز بالمربى
حليب بالقهوة حبز لامربى
حليب + قهوة + زبدة + حلويات + خبز محسن
حليب,خبز بالمعجون
كأس حليب ,قهوة ,كرواسون 
 خبز بالشكولاطة+حليب بالقهوة 
كاس حليب,مادلين,كرواسون,كاس حليب,كرواسون
حليب,قهوة,هلاليات
هلاليات,حليب بالقهوة
كاس حليب بالقهوة,هلاليات
حليب,قهوة,هلاليات,مربى ,جبن
حليب مع القهوة,حلويات جافة
كاس حليب,هلاليات,كاس قهوة حليب,هلاليات
حليب,قهوة,سكر,هلاليات
قهوة مع الحليب,هلاليات,خبز,معجون
gâteau sec,حليب بالقهوة,بيسكويت جافة
كأس قهوة,كأس حليب,كأس حليب بالشوكولاطة,حلويات
مربى,حلويات,كأس قهوة,كاس حايب مع الشكولا,خبز,كاس حليب بالشكولاطة ,خبز,مربى,حلويات,كأس قهوة
حليب بلقهوة ,مادلان 
حليب,حلويات,حليب,حلويات
café au lait ,croissant ou petit pain ,confiture 
حليب با

---
##  Appendix — Manual Meal Cleaning Workflow *(optional)*

The automated `clean_arabic_text` function handles most normalization.
However, for the highest embedding quality, a **manual review-and-remap** step can further
unify spelling variants that the regex cleaner cannot catch (e.g., two different dish names that
refer to the same food).

The three cells below implement a lightweight workflow for this:
1. Extract all unique raw meal tokens.
2. Build a mapping DataFrame with a blank `cleaned_meal` column to fill in manually.
3. Apply an initial automatic clean to pre-populate the mapping and reduce manual effort.

> This workflow is **not required** to run the pipeline. It produces a `mapping_df` that can be
used to override the automated cleaning if desired.


#### A.1 — Extract Unique Raw Meal Values

Collects all distinct non-null meal strings from `menu_df` after stripping whitespace.
The count gives an upper bound on the number of manual corrections needed.


In [ ]:
# Step 1: extract unique raw meal values
unique_meals = (
    dfs["menu_df"]["meal"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"Number of unique meals: {len(unique_meals)}")

Number of unique meals: 67647


### 2. Create Mapping Dataset

Creates a two-column DataFrame: the original `meal` string and an empty `cleaned_meal` column.
Meals are sorted alphabetically for easier manual review.
The `cleaned_meal` column is meant to be filled in (manually or programmatically) and then
applied back to `menu_df` via a left join on `meal`.


In [ ]:
# Step 2: create mapping dataframe
mapping_df = pd.DataFrame({
    "meal": sorted(unique_meals),
    "cleaned_meal": ""
})

mapping_df.head()

,meal,cleaned_meal
0,"""مسفوف "" تمر - زبيب ,حليب ,ماء",
1,"""مسفوف "" تمر - زبيب ,حليب,ماء",
2,"""مسفوف "" تمر - زبيب,حليب,ماء",
3,"""مسفوف ""تمر-زبيب,حليب,ماء",
4,"''سلطة أرز'' ماسيدوان,غراتان خضار,سردين ,عصير ...",


### 3. Apply Initial Automatic Cleaning (optional but recommended)
This pre-fills cleaned_meal to reduce manual work.

`basic_clean` applies NFKC unicode normalization plus the same Arabic letter and diacritic rules
as the main `clean_arabic_text` function. Running it here pre-populates `mapping_df['cleaned_meal']`,
so a human reviewer only needs to correct entries where the automatic result is still ambiguous.


In [ ]:
import re
import unicodedata

def basic_clean(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)
    text = re.sub(r"[ًٌٍَُِّْ]", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

mapping_df["cleaned_meal"] = mapping_df["meal"].apply(basic_clean)